In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import optuna
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

dataset = datasets.ImageFolder("/kaggle/input/datasets/project5sanju/normal-train-dataset/RESIZED ALL",transform=transform)
targets = dataset.targets
class_names = dataset.classes
num_classes = len(class_names)

train_idx, val_idx = train_test_split(np.arange(len(targets)),test_size=0.2,stratify=targets,random_state=42)
train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
train_targets = np.array(targets)[train_idx]
class_counts = np.bincount(train_targets)


from sklearn.utils.class_weight import compute_class_weight

class_weights = 1.0 / np.sqrt(class_counts)
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights = torch.tensor(class_weights,dtype=torch.float32).to(device)

print("Class Weights:", class_weights)


batch_size = 16
dropout_rate = 0.48
fc_lr = 0.00002
layer4_lr = 0.00001
weight_decay = 0.01
label_smoothing = 0.046
scheduler_factor = 0.5
epochs = 10
patience = 3


train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False)



model = models.resnet50(pretrained=True)
in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(dropout_rate),
    nn.Linear(512, num_classes))

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

for param in model.layer4[2:].parameters():
    param.requires_grad = True

model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights,label_smoothing=label_smoothing)


optimizer = optim.Adam([{"params": model.fc.parameters(),"lr": fc_lr},
    {"params": model.layer4.parameters(),"lr": layer4_lr}],
weight_decay=weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=scheduler_factor,patience=3)

best_val_acc = 0
best_epoch = 0
counter = 0
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(epochs):

    model.train()

    train_correct = 0
    train_total = 0
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        _, preds = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += preds.eq(labels).sum().item()
    train_acc = train_correct / train_total


    model.eval()

    val_correct = 0
    val_total = 0
    val_loss = 0
    val_true = []
    val_pred = []

    with torch.no_grad():

        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_true.extend(labels.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()
    val_acc = val_correct / val_total
    val_loss /= len(val_loader)

    
    train_losses.append(train_loss / len(train_loader))
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        counter = 0
        torch.save(model.state_dict(),"best_resnet_manual.pth")
    else:
        counter += 1
    if counter >= patience:
        break

    print(f"Epoch {epoch+1} | "f"Train Acc: {train_acc:.4f} | "f"Val Acc: {val_acc:.4f}")
print(f"\nBest Validation Accuracy: {best_val_acc:.4f}")


print("\nLoading Best Saved Model...")


model = models.resnet50(pretrained=True)
in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(dropout_rate),
    nn.Linear(512, num_classes))

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

model = model.to(device)
model.load_state_dict(torch.load("best_resnet_manual.pth"))
model.eval()


Class Weights: tensor([0.8709, 1.1008, 0.7535, 1.6443, 0.7508, 0.3085, 1.5712],
       device='cuda:0')


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1 | Train Acc: 0.6594 | Val Acc: 0.6981
Epoch 2 | Train Acc: 0.7104 | Val Acc: 0.7295
Epoch 3 | Train Acc: 0.7425 | Val Acc: 0.7527
Epoch 4 | Train Acc: 0.7564 | Val Acc: 0.7752
Epoch 5 | Train Acc: 0.7675 | Val Acc: 0.7711
Epoch 6 | Train Acc: 0.7839 | Val Acc: 0.7752
Epoch 7 | Train Acc: 0.7950 | Val Acc: 0.7794
Epoch 8 | Train Acc: 0.8031 | Val Acc: 0.7794
Epoch 9 | Train Acc: 0.8067 | Val Acc: 0.7841
Epoch 10 | Train Acc: 0.8143 | Val Acc: 0.7817

Best Validation Accuracy: 0.7841

Loading Best Saved Model...


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [22]:
model.eval()

train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=False)

val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False)

model.load_state_dict(torch.load("best_resnet_manual.pth"))
model.eval()

train_true = []
train_pred = []

with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs,1)
        train_true.extend(labels.numpy())
        train_pred.extend(preds.cpu().numpy())
print("\n===== Train PERFORMANCE =====")
print("\nClassification Report:\n")
print(classification_report(train_true , train_pred, target_names=class_names))

print("\nConfusion Matrix:\n")
print(confusion_matrix(train_true, train_pred))


===== Train PERFORMANCE =====

Classification Report:

              precision    recall  f1-score   support

       AKIEC       0.89      0.89      0.89       524
         BCC       0.76      0.91      0.83       328
         BKL       0.71      0.74      0.73       700
          DF       0.76      0.78      0.77       147
         MEL       0.69      0.62      0.66       705
          NV       0.93      0.92      0.93      4176
        VASC       0.81      0.95      0.87       161

    accuracy                           0.87      6741
   macro avg       0.79      0.83      0.81      6741
weighted avg       0.87      0.87      0.87      6741


Confusion Matrix:

[[ 466   18   22    9    1    5    3]
 [   4  300   16    1    1    5    1]
 [  12   26  521    5   60   76    0]
 [  15    6    1  114    2    9    0]
 [   8   10   59    1  439  180    8]
 [  16   33  111   16  131 3844   25]
 [   0    2    0    4    0    2  153]]


In [23]:
test_dir = "/kaggle/input/datasets/project5sanju/test-data-for-all/RESIZED_TESTDATA"  

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [24]:

all_labels = []
all_preds = []

with torch.no_grad():

    for images, labels in test_loader:
        
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())

print("\n===== TEST PERFORMANCE =====")
print("\nClassification Report:\n")
print(classification_report(all_labels,all_preds,target_names=class_names))
print("\nConfusion Matrix:\n")
print(confusion_matrix(all_labels,all_preds))


===== TEST PERFORMANCE =====

Classification Report:

              precision    recall  f1-score   support

       AKIEC       0.76      0.82      0.79       164
         BCC       0.62      0.64      0.63       103
         BKL       0.57      0.63      0.60       219
          DF       0.55      0.38      0.45        47
         MEL       0.60      0.51      0.55       221
          NV       0.90      0.90      0.90      1306
        VASC       0.66      0.75      0.70        51

    accuracy                           0.80      2111
   macro avg       0.66      0.66      0.66      2111
weighted avg       0.79      0.80      0.79      2111


Confusion Matrix:

[[ 134    5   11    4    2    5    3]
 [   7   66    9    4    4   11    2]
 [  12   10  137    2   16   41    1]
 [   8    2    5   18    2   10    2]
 [   6    5   27    0  113   64    6]
 [   7   17   49    2   52 1173    6]
 [   2    2    2    3    0    4   38]]


In [25]:

import os
import pandas as pd
import numpy as np

from sklearn.metrics import (classification_report,confusion_matrix,accuracy_score,precision_score,recall_score,f1_score)


output_folder = "/kaggle/working/transfer_results_normal"
os.makedirs(output_folder, exist_ok=True)

print("Saving files to:")
print(output_folder)


train_labels = np.array(targets)[train_idx]
val_labels = np.array(targets)[val_idx]

image_names = [os.path.basename(path)
    for path, _ in test_dataset.samples]

predictions_df = pd.DataFrame({"image_name": image_names,
    "true_label": all_labels,
    "predicted_label": all_preds})

predictions_df.to_csv(os.path.join(output_folder, "predictions.csv"),index=False)


test_report_dict = classification_report(all_labels,all_preds,target_names=class_names,output_dict=True)

test_report_df = pd.DataFrame(test_report_dict).transpose()

test_report_df.to_csv(os.path.join(output_folder,"test_classification_report.csv"))


test_cm = confusion_matrix(all_labels,all_preds)

test_cm_df = pd.DataFrame(test_cm,
    index=[f"True_{c}" for c in class_names],
    columns=[f"Pred_{c}" for c in class_names])

test_cm_df.to_csv(os.path.join(output_folder,"test_confusion_matrix.csv"))


train_report_dict = classification_report(train_true,train_pred,target_names=class_names,output_dict=True)
train_report_df = pd.DataFrame(train_report_dict).transpose()
train_report_df.to_csv(os.path.join(output_folder,"train_classification_report.csv"))


train_cm = confusion_matrix(train_true,train_pred)

train_cm_df = pd.DataFrame(train_cm,
    index=[f"True_{c}" for c in class_names],
    columns=[f"Pred_{c}" for c in class_names])

train_cm_df.to_csv(os.path.join(output_folder,"train_confusion_matrix.csv"))


val_report_dict = classification_report(val_true,val_pred,target_names=class_names,output_dict=True)
val_report_df = pd.DataFrame(val_report_dict).transpose()
val_report_df.to_csv(os.path.join(output_folder,"validation_classification_report.csv"))
val_cm = confusion_matrix(val_true,val_pred)
val_cm_df = pd.DataFrame(val_cm,index=[f"True_{c}" for c in class_names],columns=[f"Pred_{c}" for c in class_names])
val_cm_df.to_csv(os.path.join(output_folder,"validation_confusion_matrix.csv"))


history_df = pd.DataFrame({"epoch": range(1,len(train_losses) + 1),
    "train_loss": train_losses,
    "val_loss": val_losses,
    "train_accuracy": train_accuracies,
    "val_accuracy": val_accuracies})

history_df.to_csv(os.path.join(output_folder,"epoch_history.csv"),index=False)


summary_df = pd.DataFrame({
    "metric": ["best_validation_accuracy",
        "best_epoch",
        "test_accuracy",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1_score"],
    "value": [best_val_acc,
        best_epoch,
        accuracy_score(all_labels,all_preds),
        precision_score(all_labels,all_preds,average="weighted"),
        recall_score(all_labels,all_preds,average="weighted"),
        f1_score(all_labels,all_preds,average="weighted")]})

summary_df.to_csv(os.path.join(output_folder,"summary_metrics.csv"),index=False)

class_distribution_df = pd.DataFrame({"class_name": class_names,
    "train_count": np.bincount(train_labels,minlength=len(class_names)),
    "validation_count": np.bincount(val_labels,minlength=len(class_names)),
    "test_count": np.bincount(all_labels,minlength=len(class_names))})

class_distribution_df.to_csv(os.path.join(output_folder,"class_distribution.csv"),index=False)


split_df = pd.DataFrame({"split": ["train","validation","test"],
    "count": [len(train_dataset),len(val_dataset),len(test_dataset)]})
split_df.to_csv(os.path.join(output_folder,"dataset_split_counts.csv"),index=False)



print("\nALL FILES SAVED SUCCESSFULLY")
print("\nSaved files:")
for file in os.listdir(output_folder):
    print(file)

Saving files to:
/kaggle/working/transfer_results_normal

ALL FILES SAVED SUCCESSFULLY

Saved files:
train_classification_report.csv
validation_confusion_matrix.csv
validation_classification_report.csv
dataset_split_counts.csv
test_confusion_matrix.csv
predictions.csv
test_classification_report.csv
train_confusion_matrix.csv
class_distribution.csv
epoch_history.csv
summary_metrics.csv
